### Summarizing Mani Mama Lecture

* The youtube transcripts were not really good and there were a lot of mistakes. That is why we had to download the video and use openai-whisper library to get it transcribed. Use the transcribe.py to take the MP4 files and output the transcript into a text file.

In [8]:
from langchain_ollama import ChatOllama
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_chroma import Chroma
import textwrap
import chromadb

DB_DIR = "./chromadb"
# Initialize your Chroma vector store
chroma_client = chromadb.PersistentClient(path=DB_DIR)

vector_store = Chroma(collection_name="mani_mama_collection", client=chroma_client)


def query_after_getting_matched_documents(user_query, ollama_model_name="granite4.1:3b"):
    # Create a retriever from the vector store getting top 10 similar documents
    retriever = vector_store.as_retriever(collection_name="mani_mama_collection", search_type="similarity", search_kwargs={"k": 5})

    llm = ChatOllama(model=ollama_model_name, base_url=None)
    # ConversationalRetrievalChain wraps the LLM + retriever
    chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, return_source_documents=True)

    result = chain.invoke({"question": user_query, "chat_history":[]})
    print(textwrap.fill(result["answer"], width=70))
    matching_docs = result["source_documents"]
    return matching_docs

In [9]:
query = "Why is arjuna saying his bow is slipping?" 
matching_docs = query_after_getting_matched_documents(query)
print("Matching document IDs:")
for doc in matching_docs:
    print('----------------------')
    print(textwrap.fill(str(doc.id), width=70))
    print('----------------------')


The text provided does not explicitly state why Arjuna says his bow is
slipping. It mentions that after addressing Dhritarashtra, Arjuna
dropped his bow and arrow due to overwhelming emotions of sorrow and
disbelief in the righteousness of fighting. The phrase "seeing the
whole army once" suggests a profound emotional impact or realization
during this moment, leading him to drop his bow, indicating he was not
ready to fight. However, the specific reason for saying his bow is
slipping isn't detailed in the given context.
Matching document IDs:
----------------------
videos/003.txt[0:49:00 - 0:50:00]
----------------------
----------------------
videos/005.txt[0:12:00 - 0:13:00]
----------------------
----------------------
videos/003.txt[0:48:00 - 0:49:00]
----------------------
----------------------
videos/005.txt[0:27:00 - 0:28:00]
----------------------
----------------------
videos/005.txt[0:24:00 - 0:25:00]
----------------------


In [ ]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.documents import Document

ollama_model_name="granite4.1:3b"
llm = ChatOllama(model=ollama_model_name, base_url=None)

collection = chroma_client.get_or_create_collection(name="mani_mama_collection")

# Call the vector database for matching docs
results = collection.query(
    query_texts=["Why is the symbolism of cow"],
    n_results = 5
)

m = results["documents"]
page_content = ""
for x in m:
    page_content = x[0]
    
docs = [
    Document(page_content)
]

prompt_template = ChatPromptTemplate.from_messages(
    [("system", "Write a concise summary of the following:\\n{context}")]
)

chain = create_stuff_documents_chain(llm, prompt_template)

ans = chain.invoke({'context': docs})
print(ans)

sarvopanishadogavo dogdhagopalanandana artho vatsasudhirbhokta dhugdham gitaamritam mahataya So in the previous shloka we said gitaamritaduhe. Krishna has milked the gitaamritam. When we talk of milking, by law of association we will think of the cow and the cow means there must be a calf and then there should be a her, cow, herd, all that. So what is the cow and what is the calf? That is the question here. The imagery is nicely given in the shloka. The cow is none other than sarvopanishadaha. All the Upanishads, they are the cow. kisha, kena, kata, unda, kama, andukya, all those Upanishads, they are the cow. Also the Upanishad word is sri lingam.



AttributeError: type object 'list' has no attribute 'page_content'